In [ ]:
!pip install chemprop
!pip install rdkit


In [ ]:
%cd /content

/content


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet("data/dataset.parquet")

print("SHAPE:", df.shape)
print("COLUMNS:", len(df.columns))
print(df.dtypes.value_counts())
print("\nHEAD:")
print(df.head(3))


print("\nALL COLUMNS:")
for i, c in enumerate(df.columns):
    print(i, c)




SHAPE: (65220, 213)
COLUMNS: 213
int64      109
float64    101
object       3
Name: count, dtype: int64

HEAD:
                                             smiles  label  \
0                                          OCC(S)CS    0.0   
1                           CC[N+](C)(C)c1cccc(O)c1    0.0   
2  Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@@H]1O    0.0   

                 task  split    dipole  homo_lumo  electrons     energy  \
0  bioavailability_ma  train  1.136811   4.222476       38.0 -21.039561   
1  bioavailability_ma  train  4.059394   0.733778       67.0 -36.423003   
2  bioavailability_ma  train  1.482876   3.108952      102.0 -58.970051   

   mask_1  mask_2  ...  fr_sulfide  fr_sulfonamd  fr_sulfone  \
0       1       1  ...           0             0           0   
1       1       1  ...           0             0           0   
2       1       1  ...           0             0           0   

   fr_term_acetylene  fr_tetrazole  fr_thiazole  fr_thiocyan  fr_thiophene  \
0   

In [ ]:
df = df.rename(columns={
    "mask_1": "mask_dipole",
    "mask_2": "mask_homo_lumo",
    "mask_3": "mask_electrons",
    "mask_4": "mask_energy",
})


df = df.drop_duplicates(
    subset=["smiles","task"],
    keep="first"
)

qc_cols = ["dipole", "homo_lumo", "electrons", "energy"]
mask_cols = ["mask_dipole", "mask_homo_lumo", "mask_electrons", "mask_energy"]

for c, m in zip(qc_cols, mask_cols):
    df.loc[df[m] == 0, c] = 0.0


for c, m in zip(qc_cols, mask_cols):
    bad = df[(df[m] == 0) & (df[c] != 0)]
    print(c, "bad:", len(bad))





dipole bad: 0
homo_lumo bad: 0
electrons bad: 0
energy bad: 0


In [ ]:


print("NaN:", df.isna().sum().sum())
nan_cols = df.columns[df.isna().any()]
print("NaN COLS:", nan_cols)
print(df[nan_cols].isna().sum())

NaN: 0
NaN COLS: Index([], dtype='object')
Series([], dtype: float64)


In [ ]:




rdkit_cols = [
    c for c in df.columns
    if c not in [
        "smiles","label","task","split",
        "dipole","homo_lumo","electrons","energy",
        "mask_dipole","mask_homo_lumo","mask_electrons","mask_energy",
        "success"
    ]
]

df.loc[df["success"] == 0, rdkit_cols] = 0.0
df[rdkit_cols] = df[rdkit_cols].fillna(0.0)


import numpy as np

print("NaN:", df.isna().sum().sum())
print("INF:", np.isinf(df.select_dtypes(include=np.number)).sum().sum())

df.to_parquet(
    "dataset_raw.parquet",
    engine="pyarrow",
    compression="snappy"
)



NaN: 0
INF: 0


In [ ]:
import numpy as np

print("NaN:", df.isna().sum().sum())
print("INF:", np.isinf(df.select_dtypes(include=np.number)).sum().sum())

df.to_parquet(
    "dataset_raw.parquet",
    engine="pyarrow",
    compression="snappy"
)


NaN: 0
INF: 0


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet("dataset_raw.parquet")

print("SHAPE:", df.shape)
print("NaN:", df.isna().sum().sum())
print("COLUMNS:", len(df.columns))


qc_cols = ["dipole","homo_lumo","electrons","energy"]
mask_cols = ["mask_dipole","mask_homo_lumo","mask_electrons","mask_energy"]

rdkit_cols = [
    c for c in df.columns
    if c not in [
        "smiles","label","task","split",
        *qc_cols,*mask_cols,"success"
    ]
]

print("RDKit:", len(rdkit_cols))  # musi być 200



print("NaN:", df.isna().sum().sum())

# QC check
for c, m in zip(qc_cols, mask_cols):
    bad = ((df[m] == 0) & (df[c] != 0)).sum()
    print(c, "bad:", bad)

# RDKit check
print("RDKit NaN:", df[rdkit_cols].isna().sum().sum())



SHAPE: (65127, 213)
NaN: 0
COLUMNS: 213
RDKit: 200
NaN: 0
dipole bad: 0
homo_lumo bad: 0
electrons bad: 0
energy bad: 0
RDKit NaN: 0


In [ ]:
y = df.pivot_table(index="smiles", columns="task", values="label")
mask_y = (~y.isna()).astype(int)
y = y.fillna(0)

print("Tasks:", y.shape[1])

Tasks: 13


In [ ]:
feat_df = df.groupby("smiles").first()

X_rdkit = feat_df[rdkit_cols].values
X_qc = feat_df[qc_cols].values
X_mask = feat_df[mask_cols].values

In [ ]:
print("RDKit:", X_rdkit.shape)
print("QC:", X_qc.shape)
print("MASK:", X_mask.shape)

RDKit: (40000, 200)
QC: (40000, 4)
MASK: (40000, 4)


In [ ]:
from sklearn.preprocessing import StandardScaler

# RDKit
scaler_rdkit = StandardScaler()
X_rdkit = scaler_rdkit.fit_transform(X_rdkit)

# QC (mask-aware)
for i in range(4):
    valid = X_mask[:, i] == 1
    if valid.sum() > 0:
        scaler = StandardScaler()
        X_qc[valid, i] = scaler.fit_transform(
            X_qc[valid, i].reshape(-1,1)
        ).flatten()







In [ ]:
X_tab = np.concatenate([
    X_rdkit,
    X_qc,
    X_mask
], axis=1)




In [ ]:
assert X_tab.shape[1] == 208
assert not np.isnan(X_tab).any()
assert not np.isinf(X_tab).any()

In [ ]:
print(df.columns[:15])

Index(['smiles', 'label', 'task', 'split', 'dipole', 'homo_lumo', 'electrons',
       'energy', 'mask_dipole', 'mask_homo_lumo', 'mask_electrons',
       'mask_energy', 'success', 'MaxAbsEStateIndex', 'MaxEStateIndex'],
      dtype='object')


In [ ]:
df_proc = pd.DataFrame(X_tab, index=feat_df.index)

df_proc["y"] = list(y.values)
df_proc["mask_y"] = list(mask_y.values)

df_proc.columns = df_proc.columns.astype(str)

df_proc.to_parquet(
    "dataset_processed.parquet",
    engine="pyarrow",
    compression="snappy"
)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet("dataset_processed.parquet")

print("SHAPE:", df.shape)
print("COLUMNS:", len(df.columns))

print(df.columns[:10])   # features
print(df.columns[-3:])   # y, mask_y

feature_cols = [c for c in df.columns if c not in ["y","mask_y"]]

print("FEATURE DIM:", len(feature_cols))


X = df[feature_cols].values

print("NaN:", np.isnan(X).sum())
print("INF:", np.isinf(X).sum())



print("MEAN:", np.mean(X))
print("STD:", np.std(X))


# ostatnie 4 kolumny features = mask
mask = X[:, -4:]

print("Mask unique:", np.unique(mask))


y = df["y"]

print(type(y.iloc[0]))
print(len(y.iloc[0]))



y_all = np.stack(df["y"].values)

print("Label unique:", np.unique(y_all))


mask_y = np.stack(df["mask_y"].values)

print("mask_y unique:", np.unique(mask_y))


print("X:", X.shape)
print("y:", y_all.shape)
print("mask:", mask_y.shape)


i = 0

print("X sample:", X[i][:10])
print("y sample:", y_all[i])
print("mask sample:", mask_y[i])


SHAPE: (40000, 210)
COLUMNS: 210
Index(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'], dtype='object')
Index(['207', 'y', 'mask_y'], dtype='object')
FEATURE DIM: 208
NaN: 0
INF: 0
MEAN: 0.018904326923076933
STD: 0.9946729900790596
Mask unique: [0. 1.]
<class 'numpy.ndarray'>
13
Label unique: [0. 1.]
mask_y unique: [0 1]
X: (40000, 208)
y: (40000, 13)
mask: (40000, 13)
X sample: [-2.21918566 -2.21918566 -0.79212276  0.62326158  0.02657937  0.76095962
  0.91193045  0.74637797 -0.48478712 -0.06471226]
y sample: [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
mask sample: [0 0 0 0 0 0 0 0 1 0 0 0 0]


In [ ]:
assert X.shape[1] == 208
assert y_all.shape[1] == 13
assert not np.isnan(X).any()
assert not np.isinf(X).any()
assert set(np.unique(y_all)).issubset({0,1})
assert set(np.unique(mask_y)).issubset({0,1})

In [ ]:
B = 32

z = np.random.randn(B, 300)
x = np.random.randn(B, 208)

print(z.shape[1] + x.shape[1])

508


In [ ]:
dup = df.groupby(["smiles","task"])["label"].nunique()
conflicts = dup[dup > 1]

print("Conflicting pairs BEFORE:", len(conflicts))

# FIX
df = df.drop_duplicates(
    subset=["smiles","task"],
    keep="first"
)

# 🔴 PRZELICZ JESZCZE RAZ
dup = df.groupby(["smiles","task"])["label"].nunique()
conflicts = dup[dup > 1]

print("Conflicting pairs AFTER:", len(conflicts))

Conflicting pairs BEFORE: 0
Conflicting pairs AFTER: 0
